In [1]:
# ==============================================================================
# 1. GOOGLE COLAB AUTHENTICATION
# ==============================================================================
from google.colab import auth
import gspread
from google.auth import default
import pandas as pd
import numpy as np
import requests

print("Requesting Google Drive access permissions...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("Authentication successful!\n")

# ==============================================================================
# 2. DATA INGESTION (25/26 HISTORY)
# ==============================================================================
print("Downloading performance and player data (25/26 History)...")

url_fixtures_2526 = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2025-26/fixtures.csv"
df = pd.read_csv(url_fixtures_2526)

url_teams_2526 = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2025-26/teams.csv"
df_teams = pd.read_csv(url_teams_2526)

url_players_gw_2526 = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2025-26/gws/merged_gw.csv"
df_players = pd.read_csv(url_players_gw_2526)

url_players_raw = "https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2025-26/players_raw.csv"
df_raw_players = pd.read_csv(url_players_raw)

# ==============================================================================
# 3. TRANSFORMATION: TEAM xG CALCULATION
# ==============================================================================
df_played = df[df['finished'] == True].copy()

df_players['expected_goals'] = pd.to_numeric(df_players['expected_goals'], errors='coerce').fillna(0)
xg_per_team = df_players.groupby(['fixture', 'team'])['expected_goals'].sum().reset_index()

team_dict = dict(zip(df_teams['id'], df_teams['name']))
df_played['team_h_name'] = df_played['team_h'].map(team_dict)
df_played['team_a_name'] = df_played['team_a'].map(team_dict)

df_played = pd.merge(df_played, xg_per_team, left_on=['id', 'team_h_name'], right_on=['fixture', 'team'], how='left')
df_played = df_played.rename(columns={'expected_goals': 'expected_goals_h'}).drop(columns=['fixture', 'team'])

df_played = pd.merge(df_played, xg_per_team, left_on=['id', 'team_a_name'], right_on=['fixture', 'team'], how='left')
df_played = df_played.rename(columns={'expected_goals': 'expected_goals_a'}).drop(columns=['fixture', 'team'])

df_played['expected_goals_h'] = df_played['expected_goals_h'].fillna(0)
df_played['expected_goals_a'] = df_played['expected_goals_a'].fillna(0)

# ==============================================================================
# 4. TRANSFORMATION: MOVING AVERAGES AND DELTAS (TEAM STRENGTH)
# ==============================================================================
print("Processing team performance moving averages and deltas...")

df_home = df_played[['event', 'team_h_name', 'team_h_score', 'expected_goals_h']].rename(columns={'event': 'Gameweek', 'team_h_name': 'Club', 'team_h_score': 'Goals_Scored', 'expected_goals_h': 'xG'})
df_home['Goals_Conceded'] = df_played['team_a_score']
df_home['xGA'] = df_played['expected_goals_a']

df_away = df_played[['event', 'team_a_name', 'team_a_score', 'expected_goals_a']].rename(columns={'event': 'Gameweek', 'team_a_name': 'Club', 'team_a_score': 'Goals_Scored', 'expected_goals_a': 'xG'})
df_away['Goals_Conceded'] = df_played['team_h_score']
df_away['xGA'] = df_played['expected_goals_h']

df_teams_metrics = pd.concat([df_home, df_away]).sort_values(by=['Club', 'Gameweek'])

df_teams_metrics['Avg_Attack_6G'] = df_teams_metrics.groupby('Club')['Goals_Scored'].transform(lambda x: x.rolling(window=6, min_periods=1).mean())
df_teams_metrics['Avg_xG_6G'] = df_teams_metrics.groupby('Club')['xG'].transform(lambda x: x.rolling(window=6, min_periods=1).mean())
df_teams_metrics['Avg_Defense_6G'] = df_teams_metrics.groupby('Club')['Goals_Conceded'].transform(lambda x: x.rolling(window=6, min_periods=1).mean())
df_teams_metrics['Avg_xGA_6G'] = df_teams_metrics.groupby('Club')['xGA'].transform(lambda x: x.rolling(window=6, min_periods=1).mean())

df_teams_metrics['Delta_Attack'] = df_teams_metrics['Avg_Attack_6G'] - df_teams_metrics['Avg_xG_6G']
df_teams_metrics['Delta_Defense'] = df_teams_metrics['Avg_Defense_6G'] - df_teams_metrics['Avg_xGA_6G']

team_metrics_table = df_teams_metrics.drop_duplicates(subset=['Club'], keep='last').round(2)
final_team_cols = ['Club', 'Avg_Attack_6G', 'Avg_xG_6G', 'Delta_Attack', 'Avg_Defense_6G', 'Avg_xGA_6G', 'Delta_Defense']
team_metrics_table = team_metrics_table[final_team_cols].sort_values(by='Avg_xG_6G', ascending=False)

# ==============================================================================
# 5. TRANSFORMATION: FDR FOR THE NEW SEASON (VIA OFFICIAL API WITH FALLBACK)
# ==============================================================================
print("Calculating FDR by consuming directly from the Official FPL API...")

url_api_fixtures = "https://fantasy.premierleague.com/api/fixtures/"
url_api_bootstrap = "https://fantasy.premierleague.com/api/bootstrap-static/"

response_fixtures = requests.get(url_api_fixtures)
response_bootstrap = requests.get(url_api_bootstrap)

df_fixtures_2627 = pd.DataFrame(response_fixtures.json())
df_bootstrap = response_bootstrap.json()
df_teams_2627 = pd.DataFrame(df_bootstrap['teams'])

if 'finished' in df_fixtures_2627.columns:
    df_fixtures_2627 = df_fixtures_2627[df_fixtures_2627['finished'] == False]
df_fixtures_2627 = df_fixtures_2627.dropna(subset=['event'])

if df_fixtures_2627.empty:
    print("⚠️ WARNING: The official API has not yet released the fixtures for the new season.")
    fdr_table = pd.DataFrame(columns=['Club', 'FDR_Attack', 'FDR_Defense'])
else:
    dict_teams_2627 = dict(zip(df_teams_2627['id'], df_teams_2627['name']))
    df_fixtures_2627['team_h_name'] = df_fixtures_2627['team_h'].map(dict_teams_2627)
    df_fixtures_2627['team_a_name'] = df_fixtures_2627['team_a'].map(dict_teams_2627)

    df_future_h = df_fixtures_2627[['event', 'team_h_name', 'team_a_name']].rename(columns={'event': 'GW', 'team_h_name': 'Club', 'team_a_name': 'Opponent'})
    df_future_a = df_fixtures_2627[['event', 'team_a_name', 'team_h_name']].rename(columns={'event': 'GW', 'team_a_name': 'Club', 'team_h_name': 'Opponent'})

    schedule = pd.concat([df_future_h, df_future_a]).sort_values(by=['Club', 'GW'])

    dict_xga = dict(zip(team_metrics_table['Club'], team_metrics_table['Avg_xGA_6G']))
    dict_xg = dict(zip(team_metrics_table['Club'], team_metrics_table['Avg_xG_6G']))

    schedule['FDR_Attack'] = schedule['Opponent'].map(dict_xga)
    schedule['FDR_Defense'] = schedule['Opponent'].map(dict_xg)

    promoted_xg_avg = team_metrics_table.nsmallest(4, 'Avg_xG_6G')['Avg_xG_6G'].mean()
    promoted_xga_avg = team_metrics_table.nlargest(4, 'Avg_xGA_6G')['Avg_xGA_6G'].mean()

    schedule['FDR_Attack'] = schedule['FDR_Attack'].fillna(promoted_xga_avg)
    schedule['FDR_Defense'] = schedule['FDR_Defense'].fillna(promoted_xg_avg)

    aggregated_fdr = schedule.groupby('Club').head(4).groupby('Club').agg({'FDR_Attack': 'sum', 'FDR_Defense': 'sum'}).reset_index().round(2)
    fdr_table = aggregated_fdr.sort_values(by='FDR_Attack', ascending=False)
    print("FDR Schedule processed successfully!")

# ==============================================================================
# 6. EXTRACTION AND FILTERING OF TOP PLAYERS (WITH ADVANCED STATS)
# ==============================================================================
print("Filtering and organizing the player ranking with advanced statistics...")

target_columns = [
    'id', 'web_name', 'element_type', 'minutes', 'starts', 'total_points',
    'points_per_game', 'goals_scored', 'assists', 'expected_goals',
    'expected_assists', 'clean_sheets', 'saves', 'bonus', 'bps', 'ict_index'
]
df_filtered = df_raw_players[target_columns].copy()

position_dict = {1: 'Goalkeeper', 2: 'Defender', 3: 'Midfielder', 4: 'Forward'}
df_filtered['Position'] = df_filtered['element_type'].map(position_dict)

df_filtered = df_filtered[df_filtered['minutes'] > 100].copy()

df_filtered = df_filtered.rename(columns={
    'id': 'Player_ID', 'web_name': 'Player', 'total_points': 'Points',
    'points_per_game': 'PPG', 'starts': 'Starts',
    'goals_scored': 'Goals', 'assists': 'Assists', 'expected_goals': 'xG',
    'expected_assists': 'xA', 'clean_sheets': 'Clean_Sheets',
    'saves': 'Saves', 'bonus': 'Total_Bonus', 'bps': 'Total_BPS', 'ict_index': 'ICT_Index'
})

top_gks = df_filtered[df_filtered['Position'] == 'Goalkeeper'].sort_values(by='Points', ascending=False).head(10)
top_defs = df_filtered[df_filtered['Position'] == 'Defender'].sort_values(by='Points', ascending=False).head(40)
top_mids = df_filtered[df_filtered['Position'] == 'Midfielder'].sort_values(by='Points', ascending=False).head(40)
top_fwds = df_filtered[df_filtered['Position'] == 'Forward'].sort_values(by='Points', ascending=False).head(40)

final_player_cols = [
    'Player_ID', 'Player', 'Position', 'Starts', 'Points', 'PPG',
    'Goals', 'xG', 'Assists', 'xA', 'Clean_Sheets',
    'Saves', 'Total_Bonus', 'Total_BPS', 'ICT_Index'
]
players_table = pd.concat([top_gks, top_defs, top_mids, top_fwds])[final_player_cols]

# ==============================================================================
# 7. EXTRACTION: INDIVIDUAL DATA SPLIT BY POSITION AND WITH SCORELINE
# ==============================================================================
print("Generating individual performance databases by position...")

gw_columns = [
    'element', 'name', 'position', 'round', 'minutes', 'opponent_team', 'was_home',
    'team_h_score', 'team_a_score', 'clean_sheets', 'goals_conceded',
    'total_points', 'goals_scored', 'assists', 'expected_goals', 'expected_assists',
    'creativity', 'influence', 'bps', 'saves'
]
df_gw = df_players[gw_columns].copy()

df_gw = df_gw[df_gw['minutes'] > 0].copy()

relevant_ids = players_table['Player_ID'].tolist()
df_gw = df_gw[df_gw['element'].isin(relevant_ids)].copy()

df_gw['Team_Goals'] = np.where(df_gw['was_home'] == True, df_gw['team_h_score'], df_gw['team_a_score'])
df_gw['Opp_Goals'] = np.where(df_gw['was_home'] == True, df_gw['team_a_score'], df_gw['team_h_score'])

df_gw['Team_Goals'] = df_gw['Team_Goals'].fillna(0).astype(int).astype(str)
df_gw['Opp_Goals'] = df_gw['Opp_Goals'].fillna(0).astype(int).astype(str)
df_gw['Score'] = df_gw['Team_Goals'] + " - " + df_gw['Opp_Goals']

clean_names_dict = dict(zip(players_table['Player_ID'], players_table['Player']))
df_gw['name'] = df_gw['element'].map(clean_names_dict)

gw_position_dict = {'GK': 'Goalkeeper', 'DEF': 'Defender', 'MID': 'Midfielder', 'FWD': 'Forward'}
df_gw['position'] = df_gw['position'].map(gw_position_dict)
df_gw['opponent_team'] = df_gw['opponent_team'].map(team_dict)
df_gw['was_home'] = df_gw['was_home'].map({True: 'Home', False: 'Away'})

df_gw = df_gw.rename(columns={
    'name': 'Player', 'position': 'Position', 'round': 'Gameweek',
    'minutes': 'Minutes', 'opponent_team': 'Opponent', 'was_home': 'Home_Away',
    'total_points': 'Points', 'goals_scored': 'Goals', 'assists': 'Assists',
    'expected_goals': 'xG', 'expected_assists': 'xA',
    'creativity': 'Creativity', 'influence': 'Influence', 'bps': 'BPS',
    'saves': 'Saves', 'clean_sheets': 'Clean_Sheets', 'goals_conceded': 'Goals_Conceded'
})

# ------------------------------------------------------------------------------
# SPLITTING DATAFRAMES BY POSITION WITH EXCLUSIVE COLUMNS
# ------------------------------------------------------------------------------

# GOALKEEPERS
cols_gk = ['Player', 'Gameweek', 'Home_Away', 'Opponent', 'Score', 'Minutes', 'Points', 'Clean_Sheets', 'Goals_Conceded', 'Saves', 'BPS', 'Influence']
df_gk_matches = df_gw[df_gw['Position'] == 'Goalkeeper'][cols_gk].sort_values(by=['Player', 'Gameweek'])

# DEFENDERS
cols_def = ['Player', 'Gameweek', 'Home_Away', 'Opponent', 'Score', 'Minutes', 'Points', 'Clean_Sheets', 'Goals_Conceded', 'Goals', 'Assists', 'xG', 'xA', 'BPS', 'Influence', 'Creativity']
df_def_matches = df_gw[df_gw['Position'] == 'Defender'][cols_def].sort_values(by=['Player', 'Gameweek'])

# MIDFIELDERS & FORWARDS
cols_attacking = ['Player', 'Gameweek', 'Home_Away', 'Opponent', 'Score', 'Minutes', 'Points', 'Goals', 'Assists', 'xG', 'xA', 'BPS', 'Influence', 'Creativity']
df_mid_matches = df_gw[df_gw['Position'] == 'Midfielder'][cols_attacking].sort_values(by=['Player', 'Gameweek'])
df_fwd_matches = df_gw[df_gw['Position'] == 'Forward'][cols_attacking].sort_values(by=['Player', 'Gameweek'])

print(f"Done! Goalkeepers: {len(df_gk_matches)} games | Defenders: {len(df_def_matches)} games | Midfielders: {len(df_mid_matches)} games | Forwards: {len(df_fwd_matches)} games")

# ==============================================================================
# 8. FULL EXPORT TO GOOGLE SHEETS (OPTIMIZED FUNCTION)
# ==============================================================================
SHEET_NAME = 'FPL_Data'
print(f"\nConnecting to Google Drive. Updating spreadsheet '{SHEET_NAME}'...")

try:
    spreadsheet = gc.open(SHEET_NAME)
except gspread.exceptions.SpreadsheetNotFound:
    spreadsheet = gc.create(SHEET_NAME)

def update_tab(spreadsheet, tab_name, df):
    try:
        tab = spreadsheet.worksheet(tab_name)
    except gspread.exceptions.WorksheetNotFound:
        tab = spreadsheet.add_worksheet(title=tab_name, rows="5000", cols="20")
    tab.clear()
    if not df.empty:
        tab.update([df.columns.values.tolist()] + df.values.tolist())
    print(f"-> Tab '{tab_name}' updated!")

update_tab(spreadsheet, 'Team_Metrics', team_metrics_table)
update_tab(spreadsheet, 'FDR_Schedule', fdr_table)
update_tab(spreadsheet, 'Top_Players', players_table)

update_tab(spreadsheet, 'GK_Matches', df_gk_matches)
update_tab(spreadsheet, 'DEF_Matches', df_def_matches)
update_tab(spreadsheet, 'MID_Matches', df_mid_matches)
update_tab(spreadsheet, 'FWD_Matches', df_fwd_matches)

print("\n🚀 FULL DEPLOYMENT! All tabs have been successfully uploaded.")

Requesting Google Drive access permissions...
Authentication successful!

Processing team performance moving averages and deltas...
Calculating FDR by consuming directly from the Official FPL API...
FDR Schedule processed successfully!
Filtering and organizing the player ranking with advanced statistics...
Generating individual performance databases by position...
Done! Goalkeepers: 364 games | Defenders: 1362 games | Midfielders: 1363 games | Forwards: 1189 games

Connecting to Google Drive. Updating spreadsheet 'FPL_Data'...
-> Tab 'Team_Metrics' updated!
-> Tab 'FDR_Schedule' updated!
-> Tab 'Top_Players' updated!
-> Tab 'GK_Matches' updated!
-> Tab 'DEF_Matches' updated!
-> Tab 'MID_Matches' updated!
-> Tab 'FWD_Matches' updated!

🚀 FULL DEPLOYMENT! All tabs have been successfully uploaded.
